In [ ]:
import re
import pandas as pd
from datasets import load_dataset
from concurrent.futures import ThreadPoolExecutor, as_completed
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential
from tqdm import tqdm

In [ ]:
endpoint = "https://10725804-1478-resource.services.ai.azure.com/models"
model_name = "DeepSeek-V3"
api_key = "EXyz9vSNvy1Xf4AH2PALnjufKnTKI6HX6fnysgi0UHR7Np1s1NJGJQQJ99BEACI8hq2XJ3w3AAAAACOGpOzt"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(api_key),
    api_version="2024-05-01-preview"
)

# Load dataset
ds = load_dataset("FreedomIntelligence/RAG-Instruct", split="train[80%:]")

violations = []

# Build prompt from question/answer/documents
def build_prompt(question, answer, documents):
    docs_text = "\n".join([f"{i+1}. {doc}" for i, doc in enumerate(documents)])
    return f"""You are given a question and its answer. Below are 10 documents retrieved by a system.

Your task is to assign a relevance score from 0 to 5 to each document based on how useful it is in answering the question.

Respond as a numbered list like:
1. 3
2. 0
...

### Question:
{question}

### Answer:
{answer}

### Documents:
{docs_text}
"""

# Parse model output
def parse_scores(text):
    scores = {}
    matches = re.findall(r"(\d+)[\.\:\)]\s*([0-5])", text)
    for doc_id, score in matches:
        scores[int(doc_id)-1] = int(score)
    return [scores.get(i, 0) for i in range(10)]

# Score a single item with DeepSeek
def score_with_deepseek(index, item):
    question = item["question"]
    answer = item["answer"]
    docs = item["documents"]
    prompt = build_prompt(question, answer, docs)

    try:
        response = client.complete(
            messages=[
                SystemMessage(content="You are a helpful assistant."),
                UserMessage(content=prompt),
            ],
            max_tokens=1024,
            temperature=0.3,
            top_p=0.8,
            model=model_name
        )

        output = response.choices[0].message.content
        scores = parse_scores(output)

        results = []
        for i, score in enumerate(scores):
            results.append({
                "question": question,
                "answer": answer,
                "doc_id": i,
                "document": docs[i],
                "relevance_score": score
            })
        return results

    except Exception as e:
        print(f"[{index}] Error: {e}")
        print(f"Violating policy question: {question}")
        violations.append(index)
        results.append(None)
        return []

# Run in parallel
all_results = []
with ThreadPoolExecutor(max_workers=10) as executor:
    futures = {executor.submit(score_with_deepseek, idx, item): idx for idx, item in enumerate(ds)}
    for future in tqdm(as_completed(futures), total=len(futures)):
        result = future.result()
        all_results.extend(result)

# Final ranking
df = pd.DataFrame(all_results)
df["rank"] = df.groupby("question")["relevance_score"].rank(method="dense", ascending=False)


In [ ]:
groups = [df.iloc[i:i+10] for i in range(0, len(df), 10)]

result = []
for group in groups:
    tuples = [(row['relevance_score'], row['doc_id']) for _, row in group.iterrows()]
    sorted_tuples = sorted(tuples, key=lambda x: x[0], reverse=True)
    result.append(sorted_tuples)

In [ ]:
np.save(f"/kaggle/working/LLM-teacher_doc_rankings_DeepseekV3_0-40.npy", result)

In [ ]:
np.save(f"/kaggle/working/violations.npy", violations)

In [ ]:
result